In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze_raw AS 
SELECT  CAST(unbase64(key) AS string) as decoded_key,offset,partition,timestamp,topic,CAST(unbase64(value) AS string) AS decoded_value
FROM read_files(
    "/Volumes/workspace/default/lab_data/lab6_dataset/",
    format => "json"
);

SELECT * FROM kafka_events_bronze_raw;

In [0]:
%sql
SELECT schema_of_json(value) from kafka_events_bronze_raw;

In [0]:
%sql
CREATE OR REPLACE TABLE kafka_events_bronze AS
SELECT  *EXCEPT(decoded_value),from_json(decoded_value,'STRUCT<device: STRING, event_name: STRING, event_previous_timestamp: BIGINT, event_timestamp: BIGINT, geo: STRUCT<city: STRING, state: STRING>, items: ARRAY<STRING>, traffic_source: STRING, user_first_touch_timestamp: BIGINT, user_id: STRING>') AS decoded_value
FROM kafka_events_bronze_raw;

SELECT * FROM kafka_events_bronze;

In [0]:
%sql
SELECT 
  decoded_value,
  decoded_value.device,
  decoded_value.traffic_source,
  decoded_value.geo,       ----- Contains another JSON formatted string
  decoded_value.items      ----- Contains a nested-array of JSON formatted strings
FROM kafka_events_bronze


In [0]:
%sql 
COPY INTO historical_users_bronze_ctas_rf
FROM (
  SELECT 
  decoded_key,
  explode_outer(decoded_value.items) AS item_in_array,
  decoded_value.items 
FROM kafka_events_bronze

)


In [0]:
%sql
SELECT
    decoded_key,
    size(decoded_value.items) AS item_count
FROM kafka_events_bronze
ORDER BY item_count DESC
LIMIT 20;

In [0]:
%sql
DESCRIBE EXTENDED  kafka_events_bronze;

In [0]:
%sql
WITH events AS (
    SELECT
        101 AS order_id,
        parse_json('{
            "device": "Android",
            "items": [
                {"item_id": "P1", "quantity": 2},
                {"item_id": "P2", "quantity": 3}
            ]
        }') AS event_data
)

SELECT
    e.order_id,
    v.pos,
    v.key,
    v.value
FROM events AS e,
LATERAL variant_explode(e.event_data:items) AS v;

In [0]:
%sql
SELECT
    b.decoded_key,
    v.pos,
    v.key,
    v.value,
    schema_of_variant(v.value) AS element_type
FROM kafka_events_bronzet AS b,
LATERAL variant_explode(
    b.json_variant_value:items
) AS v
LIMIT 20;

In [0]:
%sql
merge into silver as s
using (
    with cte(transaction_id,status,first_value,event_version) as(
    select *,roll_number() over(partition by transaction_id order by event_version desc) as first_value 
    from silver
    )
    select transaction_id,status,event_version
    from cte
    where first_value = 1
) as b
on s.transaction_id = b.transaction_id
when matched 
    AND b.event_version > s.event_version
THEN UPDATE SET
  s.status = b.status,
  s.updated_at = b.updated_at,
  s.event_version = b.event_version;


